### MILESTONE 3 – Prepare Rejected & Unemployment Data for Merging

Your goal in this milestone is:

1-  Clean both datasets
2- Create a common merge key (year_month)
3- Keep only useful, defensible variables
4- Save clean versions for safe merging

You are NOT merging yet. Only preparing.

### Step 1: Mount Drive & Define Paths

In [1]:
from google.colab import drive
import pandas as pd

# Mount Google Drive
drive.mount('/content/drive')

# Define project paths
PROJECT_ROOT = "/content/drive/MyDrive/LOAN_DEFAULT_RISK/"
RAW_PATH = PROJECT_ROOT + "data/raw/"
PROCESSED_PATH = PROJECT_ROOT + "data/processed/"

# File paths
ACCEPTED_FILE = PROCESSED_PATH + "accepted_clean_model.csv"
REJECTED_FILE = RAW_PATH + "rejected_2007_to_2018Q4.csv"
UNEMP_FILE = RAW_PATH + "ca_unemployment_seasonally_adjusted.xlsx"



Mounted at /content/drive


### PART A — REJECTED DATA (Loan Applications)
##### Step A1 — Select and rename useful columns

In [ ]:
rejected_cols_map = {
    'Amount Requested': 'loan_amnt',
    'Application Date': 'issue_d',
    'Loan Title': 'title',
    'Risk_Score': 'risk_score',
    'Debt-To-Income Ratio': 'dti',
    'Zip Code': 'zip_code',
    'State': 'addr_state',
    'Employment Length': 'emp_length',
    'Policy Code': 'policy_code'
}

rejected_model_df = rejected_df[list(rejected_cols_map.keys())].copy()
rejected_model_df.rename(columns=rejected_cols_map, inplace=True)

print(rejected_model_df.columns)



Index(['loan_amnt', 'issue_d', 'title', 'risk_score', 'dti', 'zip_code',
       'addr_state', 'emp_length', 'policy_code'],
      dtype='object')


### Step 2: Convert types FIRST (important)

In [ ]:
rejected_df.columns


Index(['Amount Requested', 'Application Date', 'Loan Title', 'Risk_Score',
       'Debt-To-Income Ratio', 'Zip Code', 'State', 'Employment Length',
       'Policy Code'],
      dtype='object')

In [ ]:
# Date
rejected_model_df['issue_d'] = pd.to_datetime(rejected_model_df['issue_d'], errors='coerce')

# Remove % and convert DTI
rejected_model_df['dti'] = rejected_model_df['dti'].astype(str).str.replace('%','')
rejected_model_df['dti'] = pd.to_numeric(rejected_model_df['dti'], errors='coerce')

# Numeric columns
rejected_model_df['loan_amnt'] = pd.to_numeric(rejected_model_df['loan_amnt'], errors='coerce')
rejected_model_df['risk_score'] = pd.to_numeric(rejected_model_df['risk_score'], errors='coerce')




In [ ]:
rejected_model_df[['loan_amnt','dti','risk_score']].dtypes


,0
loan_amnt,float64
dti,float64
risk_score,float64


### 🔹 Step 3: Handle missing values

In [ ]:
num_cols = ['loan_amnt', 'dti', 'risk_score']
for col in num_cols:
    rejected_model_df[col] = rejected_model_df[col].fillna(rejected_model_df[col].median())

cat_cols = ['title', 'zip_code', 'addr_state', 'emp_length', 'policy_code']
for col in cat_cols:
    rejected_model_df[col] = rejected_model_df[col].fillna(rejected_model_df[col].mode()[0])


### Step 4: Create merge key

In [ ]:
rejected_model_df['year_month'] = rejected_model_df['issue_d'].dt.to_period('M')


### Step 5: Monthly aggregation (CRITICAL)

In [ ]:
rejected_monthly = rejected_model_df.groupby('year_month').agg(
    rejected_applications=('loan_amnt','count'),
    rejected_avg_amount=('loan_amnt','mean'),
    rejected_avg_dti=('dti','mean'),
    rejected_avg_risk_score=('risk_score','mean')
).reset_index()

print(rejected_monthly.head())
print(rejected_monthly.shape)


  year_month  rejected_applications  rejected_avg_amount  rejected_avg_dti  \
0    2007-05                     49          6701.020408         14.232449   
1    2007-06                    289          6954.325260        108.074048   
2    2007-07                    266          6875.000000         20.222143   
3    2007-08                    222          8152.702703        372.732523   
4    2007-09                    418          7209.150718        596.579952   

   rejected_avg_risk_score  
0               611.448980  
1               537.941176  
2               530.699248  
3               537.590090  
4               532.234450  
(140, 5)


### Step 6: Save

In [ ]:
rejected_monthly.to_csv(PROCESSED_PATH + "rejected_monthly_clean.csv", index=False)


In [ ]:
rejected_monthly.head()
rejected_monthly.shape
print(rejected_monthly)


    year_month  rejected_applications  rejected_avg_amount  rejected_avg_dti  \
0      2007-05                     49          6701.020408         14.232449   
1      2007-06                    289          6954.325260        108.074048   
2      2007-07                    266          6875.000000         20.222143   
3      2007-08                    222          8152.702703        372.732523   
4      2007-09                    418          7209.150718        596.579952   
..         ...                    ...                  ...               ...   
135    2018-08                 898812         13434.220922        108.329132   
136    2018-09                 826643         13732.497280         99.147722   
137    2018-10                 916557         13701.502302         91.727199   
138    2018-11                 843496         12977.951541        102.377904   
139    2018-12                 834369         12326.704651         88.726030   

     rejected_avg_risk_score  
0       

What you’ve successfully achieved

Reduced 27.6 million rows → 140 monthly records

Created meaningful macro features:

rejected_applications

rejected_avg_amount

rejected_avg_dti

rejected_avg_risk_score

Built a clean, merge-safe dataset using year_month

This is a very strong Milestone-3 outcome.

# Unemployment dataset
### UNEMPLOYMENT DATA – CLEAN PIPELINE
#####✅ Step 1: Load correctly (skip metadata)

In [ ]:
unemp_raw = pd.read_excel(UNEMP_FILE, skiprows=9)

print(unemp_raw.head())


  Unnamed: 0 Unnamed: 1                      Unnamed: 2  \
0       Year     Period  labor force participation rate   
1       2015        Jan                            62.4   
2       2015        Feb                            62.3   
3       2015        Mar                            62.3   
4       2015        Apr                            62.3   

                    Unnamed: 3   Unnamed: 4  Unnamed: 5    Unnamed: 6  \
0  employment-population ratio  labor force  employment  unemployment   
1                         58.1     18818212    17534241       1283971   
2                         58.2     18826972    17560173       1266799   
3                         58.2     18838028    17587613       1250415   
4                         58.2     18848322    17616628       1231694   

          Unnamed: 7  
0  unemployment rate  
1                6.8  
2                6.7  
3                6.6  
4                6.5  


/usr/local/lib/python3.12/dist-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


#### Step 2: Fix headers

In [ ]:
unemp_raw.columns = unemp_raw.iloc[0]
unemp_df = unemp_raw.drop(index=0).reset_index(drop=True)

print(unemp_df.head())
print(unemp_df.columns)


0  Year Period labor force participation rate employment-population ratio  \
0  2015    Jan                           62.4                        58.1   
1  2015    Feb                           62.3                        58.2   
2  2015    Mar                           62.3                        58.2   
3  2015    Apr                           62.3                        58.2   
4  2015    May                           62.3                        58.3   

0 labor force employment unemployment unemployment rate  
0    18818212   17534241      1283971               6.8  
1    18826972   17560173      1266799               6.7  
2    18838028   17587613      1250415               6.6  
3    18848322   17616628      1231694               6.5  
4    18852828   17643698      1209130               6.4  
Index(['Year', 'Period', 'labor force participation rate',
       'employment-population ratio', 'labor force', 'employment',
       'unemployment', 'unemployment rate'],
      dtype='objec

#### ✅ Step 3: Standardize column names

In [ ]:
unemp_df.columns = (
    unemp_df.columns.str.strip()
    .str.lower()
    .str.replace(" ", "_")
    .str.replace("-", "_")
)


Expected important columns:

year

period

unemployment_rate

### ✅ Step 4: Build date and merge key

In [ ]:
unemp_df['year'] = pd.to_numeric(unemp_df['year'], errors='coerce')

unemp_df['date'] = pd.to_datetime(
    unemp_df['year'].astype(str) + "-" + unemp_df['period'],
    errors='coerce'
)

unemp_df['year_month'] = unemp_df['date'].dt.to_period('M')


/tmp/ipython-input-926991499.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  unemp_df['date'] = pd.to_datetime(


### ✅ Step 5: Keep only needed variables

In [ ]:
# View column names
print("Columns:", unemp_df.columns.tolist())

# View first rows
unemp_df.head()

# Data types and missing values
unemp_df.info()

# Missing value count per column
unemp_df.isna().sum()


Columns: ['year', 'period', 'labor_force_participation_rate', 'employment_population_ratio', 'labor_force', 'employment', 'unemployment', 'unemployment_rate', 'date', 'year_month']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 131 entries, 0 to 130
Data columns (total 10 columns):
 #   Column                          Non-Null Count  Dtype         
---  ------                          --------------  -----         
 0   year                            131 non-null    int64         
 1   period                          131 non-null    object        
 2   labor_force_participation_rate  130 non-null    object        
 3   employment_population_ratio     130 non-null    object        
 4   labor_force                     130 non-null    object        
 5   employment                      130 non-null    object        
 6   unemployment                    130 non-null    object        
 7   unemployment_rate               130 non-null    object        
 8   date                         

,0
0,
year,0
period,0
labor_force_participation_rate,1
employment_population_ratio,1
labor_force,1
employment,1
unemployment,1
unemployment_rate,1
date,0


####Final clean unemployment modeling table (approved)

In [ ]:
# Keep only what we need
unemp_model_df = unemp_df[['year_month', 'unemployment_rate']].copy()

# Convert to numeric
unemp_model_df['unemployment_rate'] = pd.to_numeric(
    unemp_model_df['unemployment_rate'], errors='coerce'
)

# Fill missing value
unemp_model_df['unemployment_rate'] = unemp_model_df['unemployment_rate'].fillna(
    unemp_model_df['unemployment_rate'].median()
)

# Ensure one row per month
unemp_model_df = unemp_model_df.drop_duplicates('year_month')

# Final verification
print(unemp_model_df.info())
print(unemp_model_df.head())
print(unemp_model_df.tail())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 131 entries, 0 to 130
Data columns (total 2 columns):
 #   Column             Non-Null Count  Dtype    
---  ------             --------------  -----    
 0   year_month         131 non-null    period[M]
 1   unemployment_rate  131 non-null    float64  
dtypes: float64(1), period[M](1)
memory usage: 2.2 KB
None
0 year_month  unemployment_rate
0    2015-01                6.8
1    2015-02                6.7
2    2015-03                6.6
3    2015-04                6.5
4    2015-05                6.4
0   year_month  unemployment_rate
126    2025-07                5.5
127    2025-08                5.5
128    2025-09                5.6
129    2025-10                5.2
130    2025-11                5.5


### step 7 save the fiel

In [ ]:
unemp_model_df.to_csv(PROCESSED_PATH + "ca_unemployment_clean.csv", index=False)


### When this finishes, Milestone-3 is COMPLETE

We have now have:

* accepted_clean_model.csv
*rejected_monthly_clean.csv
* ca_unemployment_clean.csv

These three are your gold datasets.

# Milestone-4: Integration + EDA

Next we will:

### Milestone-4 workflow



1.   Load:
*   accepted_clean_model.csv

* rejected_monthly_clean.csv

* ca_unemployment_clean.csv

2. Merge macro features into accepted loans

3. Validate:

* Row counts

* Target distribution

* Missingness after merge
Begin:

4. EDA

* Feature engineering

* Correlation with target_default

### Milestone-4 Step 1: Load all datasets

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROCESSED_PATH = "/content/drive/MyDrive/LOAN_DEFAULT_RISK/data/processed/"

# Check files in the folder
import os
os.listdir(PROCESSED_PATH)



Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


['accepted_checked.csv',
 'column_classification_partial.csv',
 'column_classification.csv',
 'rejected_monthly_clean.csv',
 'ca_unemployment_clean.csv',
 'accepted_clean_model.csv',
 'accepted_clean_model.parquet']

In [ ]:
import pandas as pd

# -------------------------------
# Paths
# -------------------------------
PROCESSED_PATH = "/content/drive/MyDrive/LOAN_DEFAULT_RISK/data/processed/"

# Files
ACCEPTED_FILE = PROCESSED_PATH + "accepted_clean_model.parquet"
REJECTED_FILE = PROCESSED_PATH + "rejected_monthly_clean.csv"
UNEMP_FILE   = PROCESSED_PATH + "ca_unemployment_clean.csv"

# -------------------------------
# Step 1: Load datasets
# -------------------------------
# Accepted dataset (Parquet)
accepted_df = pd.read_parquet(ACCEPTED_FILE)
print("Accepted shape:", accepted_df.shape)

# Rejected dataset (monthly aggregated)
rejected_df = pd.read_csv(REJECTED_FILE)
print("Rejected shape:", rejected_df.shape)

# Unemployment dataset (clean)
unemp_df = pd.read_csv(UNEMP_FILE)
print("Unemployment shape:", unemp_df.shape)

# -------------------------------
# Step 2: Merge Rejected dataset
# -------------------------------
# Merge on 'year_month'
merged_df = pd.merge(
    accepted_df,
    rejected_df,
    how='left',
    on='year_month'
)
print("After merging Rejected data:", merged_df.shape)

# -------------------------------
# Step 3: Merge Unemployment dataset
# -------------------------------
# Merge on 'year_month'
merged_df = pd.merge(
    merged_df,
    unemp_df,
    how='left',
    on='year_month'
)
print("After merging Unemployment data:", merged_df.shape)

# -------------------------------
# Step 4: Save final modeling-ready dataset
# -------------------------------
# Save as Parquet for speed
merged_df.to_parquet(PROCESSED_PATH + "accepted_modeling_ready.parquet", index=False)
print("Modeling-ready dataset saved successfully!")

# Optional: Quick check
merged_df.head()
merged_df.info()



Accepted shape: (1369566, 2008)
Rejected shape: (140, 5)
Unemployment shape: (131, 2)


KeyError: 'year_month'

In [ ]:
print(rejected_df.columns.tolist())



['year_month', 'rejected_applications', 'rejected_avg_amount', 'rejected_avg_dti', 'rejected_avg_risk_score']


In [ ]:
import pandas as pd

PROCESSED_PATH = "/content/drive/MyDrive/LOAN_DEFAULT_RISK/data/processed/"

# Accepted dataset (cleaned from Milestone 2)
accepted_df = pd.read_csv(PROCESSED_PATH + "accepted_clean_model.csv", low_memory=False)
print("Accepted shape:", accepted_df.shape)

# Rejected dataset (cleaned monthly)
rejected_df = pd.read_csv(PROCESSED_PATH + "rejected_monthly_clean.csv", low_memory=False)
print("Rejected shape:", rejected_df.shape)

# Unemployment dataset
unemp_df = pd.read_csv(PROCESSED_PATH + "unemp_clean.csv", low_memory=False)
print("Unemployment shape:", unemp_df.shape)


In [ ]:
# Accepted dataset
if 'year_month' not in accepted_df.columns:
    accepted_df['year_month'] = pd.to_datetime(accepted_df['issue_d'], errors='coerce').dt.to_period('M')

# Rejected dataset
if 'year_month' not in rejected_df.columns:
    rejected_df['year_month'] = pd.to_datetime(rejected_df['issue_d'], errors='coerce').dt.to_period('M')

# Unemployment dataset (should already have it)
unemp_df['year_month'] = pd.to_datetime(unemp_df['year_month'].astype(str)).dt.to_period('M')


In [ ]:
# Merge accepted + rejected (e.g., only loan_amnt, Risk_Score from rejected)
merged_df = accepted_df.merge(
    rejected_df[['year_month', 'loan_amnt', 'Risk_Score']],
    on='year_month',
    how='left'
)

# Merge with unemployment
merged_df = merged_df.merge(
    unemp_df[['year_month', 'unemployment_rate']],
    on='year_month',
    how='left'
)

print("Merged dataset shape:", merged_df.shape)
merged_df.head()


In [ ]:
import pandas as pd

PROCESSED_PATH = "/content/drive/MyDrive/LOAN_DEFAULT_RISK/data/processed/"

accepted_df = pd.read_csv(PROCESSED_PATH + "accepted_clean_model.csv", low_memory=False)
accepted_sample = accepted_df.sample(n=50000, random_state=42)  # 50k rows
accepted_sample.to_parquet(PROCESSED_PATH + "accepted_sample_50k.parquet", index=False)
print("Accepted sample saved:", accepted_sample.shape)


Accepted sample saved: (50000, 2008)


In [ ]:
rejected_df = pd.read_csv(PROCESSED_PATH + "rejected_monthly_clean.csv", low_memory=False)
rejected_sample = rejected_df.sample(n=50000, random_state=42)  # 50k rows
rejected_sample.to_parquet(PROCESSED_PATH + "rejected_sample_50k.parquet", index=False)
print("Rejected sample saved:", rejected_sample.shape)


ValueError: Cannot take a larger sample than population when 'replace=False'

In [ ]:
unemp_df = pd.read_csv(PROCESSED_PATH + "unemp_clean.csv")
unemp_df['unemployment_rate'] = pd.to_numeric(unemp_df['unemployment_rate'], errors='coerce')
unemp_df['unemployment_rate'].fillna(unemp_df['unemployment_rate'].median(), inplace=True)
unemp_df['year_month'] = pd.to_datetime(unemp_df['year_month'].astype(str)).dt.to_period('M')
unemp_df.to_parquet(PROCESSED_PATH + "unemp_clean.parquet", index=False)
print("Unemployment dataset saved.")


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/LOAN_DEFAULT_RISK/data/processed/unemp_clean.csv'

In [ ]:
import pandas as pd

PROCESSED_PATH = "/content/drive/MyDrive/LOAN_DEFAULT_RISK/data/processed/"

# -------------------------------
# 1️⃣ Accepted dataset
accepted_df = pd.read_parquet(PROCESSED_PATH + "accepted_clean_model.parquet")
accepted_sample = accepted_df.sample(n=50000, random_state=42)  # 50k rows
accepted_sample.to_parquet(PROCESSED_PATH + "accepted_sample_50k.parquet", index=False)
print("Accepted sample shape:", accepted_sample.shape)



Accepted sample shape: (50000, 2008)


In [ ]:
# 2️⃣ Rejected dataset
rejected_df = pd.read_csv(PROCESSED_PATH + "rejected_monthly_clean.csv", low_memory=False)
rejected_sample = rejected_df.sample(n=50000, random_state=42)
rejected_sample.to_parquet(PROCESSED_PATH + "rejected_sample_50k.parquet", index=False)
print("Rejected sample shape:", rejected_sample.shape)

ValueError: Cannot take a larger sample than population when 'replace=False'

In [ ]:
import pandas as pd

UNEMP_FILE = "/content/drive/MyDrive/LOAN_DEFAULT_RISK/data/processed/ca_unemployment_clean.csv"

# Load the unemployment dataset
unemp_df = pd.read_csv(UNEMP_FILE)

# Check shape (rows, columns)
print("Unemployment dataset shape:", unemp_df.shape)

# Show first few rows
print(unemp_df.head())

# Optional: get info and missing values
print(unemp_df.info())
print("\nMissing values per column:\n", unemp_df.isna().sum())


Unemployment dataset shape: (131, 2)
  year_month  unemployment_rate
0    2015-01                6.8
1    2015-02                6.7
2    2015-03                6.6
3    2015-04                6.5
4    2015-05                6.4
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 131 entries, 0 to 130
Data columns (total 2 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   year_month         131 non-null    object 
 1   unemployment_rate  131 non-null    float64
dtypes: float64(1), object(1)
memory usage: 2.2+ KB
None

Missing values per column:
 year_month           0
unemployment_rate    0
dtype: int64


In [ ]:
unemp_df['year_month'] = pd.to_datetime(unemp_df['year_month']).dt.to_period('M')


In [ ]:
import pandas as pd

# File paths
ACCEPTED_FILE = "/content/drive/MyDrive/LOAN_DEFAULT_RISK/data/processed/accepted_sample_50k.parquet"
REJECTED_FILE = "/content/drive/MyDrive/LOAN_DEFAULT_RISK/data/processed/rejected_monthly_clean.csv"
UNEMP_FILE   = "/content/drive/MyDrive/LOAN_DEFAULT_RISK/data/processed/ca_unemployment_clean.csv"

PROCESSED_PATH = "/content/drive/MyDrive/LOAN_DEFAULT_RISK/data/processed/"


In [ ]:
# Accepted sample
accepted_df = pd.read_parquet(ACCEPTED_FILE)
print("Accepted sample shape:", accepted_df.shape)

# Rejected dataset (all rows)
rejected_df = pd.read_csv(REJECTED_FILE, low_memory=False)
print("Rejected dataset shape:", rejected_df.shape)

# Unemployment dataset
unemp_df = pd.read_csv(UNEMP_FILE)
print("Unemployment dataset shape:", unemp_df.shape)


Accepted sample shape: (50000, 2008)
Rejected dataset shape: (140, 5)
Unemployment dataset shape: (131, 2)


In [ ]:
# Rejected: keep only columns to merge
rejected_merge_df = rejected_df[['year_month', 'loan_amnt', 'Risk_Score']]

# Unemployment: keep only columns to merge
unemp_merge_df = unemp_df[['year_month', 'unemployment_rate']]


KeyError: "['loan_amnt', 'Risk_Score'] not in index"

In [ ]:
# Merge accepted + rejected
merged_df = accepted_df.merge(
    rejected_merge_df,
    on='year_month',
    how='left',
    suffixes=('', '_rejected')
)

# Merge with unemployment
merged_df = merged_df.merge(
    unemp_merge_df,
    on='year_month',
    how='left'
)

print("Merged dataset shape:", merged_df.shape)
merged_df.head()


NameError: name 'rejected_merge_df' is not defined

In [ ]:
# 1️⃣ Prepare rejected dataset for merge
rejected_merge_df = rejected_df[['year_month', 'loan_amnt', 'Risk_Score']].copy()

# Optional: rename columns to avoid conflicts with accepted dataset
rejected_merge_df.rename(columns={
    'loan_amnt': 'loan_amnt_rejected',
    'Risk_Score': 'Risk_Score_rejected'
}, inplace=True)

# 2️⃣ Prepare unemployment dataset for merge
unemp_merge_df = unemp_df[['year_month', 'unemployment_rate']].copy()

# 3️⃣ Merge accepted + rejected
merged_df = accepted_df.merge(
    rejected_merge_df,
    on='year_month',
    how='left'
)

# 4️⃣ Merge with unemployment
merged_df = merged_df.merge(
    unemp_merge_df,
    on='year_month',
    how='left'
)

print("Merged dataset shape:", merged_df.shape)
merged_df.head()


KeyError: "['loan_amnt', 'Risk_Score'] not in index"

In [ ]:
print(rejected_df.columns.tolist())


['year_month', 'rejected_applications', 'rejected_avg_amount', 'rejected_avg_dti', 'rejected_avg_risk_score']


In [ ]:
import pandas as pd

# File paths
ACCEPTED_FILE = "/content/drive/MyDrive/LOAN_DEFAULT_RISK/data/processed/accepted_sample_50k.parquet"
REJECTED_FILE = "/content/drive/MyDrive/LOAN_DEFAULT_RISK/data/processed/rejected_monthly_clean.csv"
UNEMP_FILE   = "/content/drive/MyDrive/LOAN_DEFAULT_RISK/data/processed/ca_unemployment_clean.csv"
PROCESSED_PATH = "/content/drive/MyDrive/LOAN_DEFAULT_RISK/data/processed/"

# -------------------------
# Step 1: Load datasets
accepted_df = pd.read_parquet(ACCEPTED_FILE)
rejected_df = pd.read_csv(REJECTED_FILE, low_memory=False)
unemp_df    = pd.read_csv(UNEMP_FILE)

# -------------------------
# Step 2: Ensure year_month is period type for merging
accepted_df['year_month'] = pd.to_datetime(accepted_df['issue_d'], errors='coerce').dt.to_period('M')
rejected_df['year_month'] = pd.to_datetime(rejected_df['year_month'], errors='coerce').dt.to_period('M')
unemp_df['year_month']    = pd.to_datetime(unemp_df['year_month'], errors='coerce').dt.to_period('M')

# -------------------------
# Step 3: Prepare rejected and unemployment for merge
rejected_merge_df = rejected_df[['year_month', 'rejected_applications',
                                 'rejected_avg_amount', 'rejected_avg_dti',
                                 'rejected_avg_risk_score']].copy()

unemp_merge_df = unemp_df[['year_month', 'unemployment_rate']].copy()

# -------------------------
# Step 4: Merge datasets
merged_df = accepted_df.merge(
    rejected_merge_df,
    on='year_month',
    how='left'
).merge(
    unemp_merge_df,
    on='year_month',
    how='left'
)

# -------------------------
# Step 5: Check merged dataset
print("Merged dataset shape:", merged_df.shape)
merged_df.head()

# -------------------------
# Step 6: Save merged dataset
merged_df.to_parquet(PROCESSED_PATH + "merged_accepted_rejected_unemp_50k.parquet", index=False)
print("Merged dataset saved successfully!")


KeyError: 'issue_d'

In [ ]:
print(accepted_df.columns.tolist())


['loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'int_rate', 'installment', 'annual_inc', 'dti', 'delinq_2yrs', 'fico_range_low', 'fico_range_high', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'collections_12_mths_ex_med', 'policy_code', 'acc_now_delinq', 'tot_coll_amt', 'tot_cur_bal', 'total_rev_hi_lim', 'acc_open_past_24mths', 'avg_cur_bal', 'bc_open_to_buy', 'bc_util', 'chargeoff_within_12_mths', 'delinq_amnt', 'mo_sin_old_il_acct', 'mo_sin_old_rev_tl_op', 'mo_sin_rcnt_rev_tl_op', 'mo_sin_rcnt_tl', 'mort_acc', 'mths_since_recent_bc', 'mths_since_recent_inq', 'num_accts_ever_120_pd', 'num_actv_bc_tl', 'num_actv_rev_tl', 'num_bc_sats', 'num_bc_tl', 'num_il_tl', 'num_op_rev_tl', 'num_rev_accts', 'num_rev_tl_bal_gt_0', 'num_sats', 'num_tl_120dpd_2m', 'num_tl_30dpd', 'num_tl_90g_dpd_24m', 'num_tl_op_past_12m', 'pct_tl_nvr_dlq', 'percent_bc_gt_75', 'pub_rec_bankruptcies', 'tax_liens', 'tot_hi_cred_lim', 'total_bal_ex_mort', 'total_bc_limit', 'total_il_h

In [ ]:
# Select only issue_d columns
issue_cols = [col for col in accepted_df.columns if col.startswith('issue_d_')]

# Sum loan amounts by month
accepted_monthly = accepted_df[issue_cols].sum().reset_index()
accepted_monthly.columns = ['year_month', 'accepted_loan_count']

# Convert 'issue_d_Apr-2009' -> '2009-04' format
accepted_monthly['year_month'] = accepted_monthly['year_month'].str.replace('issue_d_', '')
accepted_monthly['year_month'] = pd.to_datetime(accepted_monthly['year_month'], format='%b-%Y').dt.to_period('M')

accepted_monthly.head()


,year_month,accepted_loan_count
0,2009-04,8
1,2010-04,26
2,2011-04,65
3,2012-04,116
4,2013-04,356


In [ ]:
# First, make sure all year_month columns are the same type
rejected_df['year_month'] = pd.to_datetime(rejected_df['year_month']).dt.to_period('M')
unemployment_df['year_month'] = pd.to_datetime(unemployment_df['year_month']).dt.to_period('M')

# Merge accepted and rejected
merged_df = pd.merge(accepted_monthly, rejected_df, on='year_month', how='outer')

# Merge the above with unemployment data
merged_df = pd.merge(merged_df, unemployment_df, on='year_month', how='outer')

# Sort by year_month
merged_df = merged_df.sort_values('year_month').reset_index(drop=True)

merged_df.head()


NameError: name 'unemployment_df' is not defined

@@@@@@@@@@@@@@@@@@@@@@@@@@@@@@


In [ ]:
accepted_df = pd.read_parquet(ACCEPTED_FILE)
rejected_df = pd.read_csv(REJECTED_FILE, low_memory=False)
unemp_df    = pd.read_csv(UNEMP_FILE)

In [ ]:
accepted_df['year_month'] = pd.to_datetime(accepted_df['issue_d'], errors='coerce').dt.to_period('M')
rejected_df['year_month'] = pd.to_datetime(rejected_df['year_month'], errors='coerce').dt.to_period('M')
unemp_df['year_month']    = pd.to_datetime(unemp_df['year_month'], errors='coerce').dt.to_period('M')


KeyError: 'issue_d'

In [ ]:
print(accepted_df.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Columns: 2008 entries, loan_amnt to application_type_Joint App
dtypes: bool(1950), float64(58)
memory usage: 115.1 MB
None


In [ ]:
# Target
y = accepted_df['target_default']  # Make sure this exists

# Predictors (drop target)
X = accepted_df.drop(columns=['target_default'])


In [ ]:
numeric_cols = X.select_dtypes(include=['float64', 'int64']).columns.tolist()
bool_cols    = X.select_dtypes(include=['bool']).columns.tolist()

print("Numeric columns:", len(numeric_cols))
print("Boolean columns:", len(bool_cols))


Numeric columns: 57
Boolean columns: 1950


In [ ]:
# Only for numeric + boolean predictors
corr = X[numeric_cols + bool_cols].corrwith(y)

# Sort by absolute correlation
corr_sorted = corr.abs().sort_values(ascending=False)
print(corr_sorted.head(20))  # Top 20 most correlated variables


/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2922: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:2923: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


loan_status_Fully Paid            0.995500
int_rate                          0.262739
loan_status_Late (31-120 days)    0.248809
term_ 60 months                   0.173280
grade_E                           0.130053
fico_range_low                    0.129856
fico_range_high                   0.129856
acc_open_past_24mths              0.103531
grade_D                           0.101778
grade_B                           0.100496
grade_F                           0.092604
num_tl_op_past_12m                0.087357
dti                               0.079453
mort_acc                          0.074625
home_ownership_MORTGAGE           0.073673
num_actv_rev_tl                   0.072578
avg_cur_bal                       0.072091
bc_open_to_buy                    0.070779
num_rev_tl_bal_gt_0               0.070775
loan_amnt                         0.069225
dtype: float64


In [ ]:
# Drop redundant columns
X = accepted_df.drop(columns=['loan_status_Fully Paid', 'loan_status_Late (31-120 days)'])

# Keep top 50 correlated features (excluding target)
top_features = corr_sorted.head(50).index.tolist()
top_features = [f for f in top_features if f not in ['loan_status_Fully Paid']]  # remove target
reduced_df = accepted_df[top_features + ['target_default']]  # keep target
print("Reduced dataset shape:", reduced_df.shape)


Reduced dataset shape: (50000, 50)


In [ ]:
import pandas as pd

# Load the dataset
accepted_checked = pd.read_csv("/content/drive/MyDrive/LOAN_DEFAULT_RISK/data/processed/accepted_clean_model.csv")

# Check info
print(accepted_checked.info())

# Check first 5 rows
print(accepted_checked.head())

# Optional: check missing values per column
print("Missing values per column:")
print(accepted_checked.isna().sum())


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 229565 entries, 0 to 229564
Columns: 2008 entries, loan_amnt to application_type_Joint App
dtypes: bool(1950), float64(58)
memory usage: 528.5 MB
None
   loan_amnt  funded_amnt  funded_amnt_inv  int_rate  installment  annual_inc  \
0     3600.0       3600.0           3600.0     13.99       123.03     55000.0   
1    24700.0      24700.0          24700.0     11.99       820.28     65000.0   
2    20000.0      20000.0          20000.0     10.78       432.66     63000.0   
3    10400.0      10400.0          10400.0     22.45       289.91    104433.0   
4    11950.0      11950.0          11950.0     13.44       405.18     34000.0   

     dti  delinq_2yrs  fico_range_low  fico_range_high  ...  \
0   5.91          0.0           675.0            679.0  ...   
1  16.06          1.0           715.0            719.0  ...   
2  10.78          0.0           695.0            699.0  ...   
3  25.37          1.0           695.0            699.0  ... 

In [ ]:
import pandas as pd

# Load dataset (you already have it, adjust path if needed)
accepted_checked = pd.read_csv("/content/drive/MyDrive/LOAN_DEFAULT_RISK/data/processed/accepted_checked.csv", low_memory=False)

# -----------------------
# 1️⃣ Select numeric columns only
num_cols = accepted_checked.select_dtypes(include=['float64','int64']).columns.tolist()

# Make sure target_default is in the list
if 'target_default' not in num_cols:
    num_cols.append('target_default')

# -----------------------
# 2️⃣ Compute correlation with target_default
corr_with_target = accepted_checked[num_cols].corr()['target_default'].sort_values(ascending=False)

# -----------------------
# 3️⃣ Show top 20 positive correlations
print("Top 20 features positively correlated with target_default:")
print(corr_with_target.head(20))

# -----------------------
# 4️⃣ Show top 20 negative correlations
print("\nTop 20 features negatively correlated with target_default:")
print(corr_with_target.tail(20))


Top 20 features positively correlated with target_default:
target_default                                1.000000
recoveries                                    0.481679
collection_recovery_fee                       0.457210
int_rate                                      0.263009
hardship_dpd                                  0.244928
out_prncp                                     0.194811
out_prncp_inv                                 0.194801
orig_projected_additional_accrued_interest    0.174669
hardship_amount                               0.165148
hardship_payoff_balance_amount                0.158215
total_rec_late_fee                            0.153840
sec_app_inq_last_6mths                        0.148801
sec_app_revol_util                            0.143684
dti_joint                                     0.141948
sec_app_collections_12_mths_ex_med            0.104177
acc_open_past_24mths                          0.099789
hardship_last_payment_amount                  0.091422
id    

In [ ]:
import pandas as pd

# Assuming accepted_checked is already loaded and target_default exists
target = 'target_default'

# Step 1: Separate numeric and boolean columns
numeric_cols = accepted_checked.select_dtypes(include=['float64', 'int']).columns.tolist()
bool_cols = accepted_checked.select_dtypes(include=['bool']).columns.tolist()

# Step 2: Compute correlation for numeric columns with target
numeric_corr = accepted_checked[numeric_cols + [target]].corr()[target].sort_values(ascending=False)
print("Top numeric features correlated with target:\n", numeric_corr.head(20))

# Step 3: For boolean columns, compute correlation with target
bool_corr = accepted_checked[bool_cols + [target]].corr()[target].sort_values(ascending=False)
print("\nTop boolean features correlated with target:\n", bool_corr.head(20))

# Step 4: Select features with correlation above a threshold
threshold = 0.05  # you can adjust
selected_numeric = numeric_corr[abs(numeric_corr) > threshold].index.tolist()
selected_bool = bool_corr[abs(bool_corr) > threshold].index.tolist()

# Combine selected features
selected_features = selected_numeric + selected_bool
print("\nNumber of selected features:", len(selected_features))

# Create reduced dataframe
reduced_df = accepted_checked[selected_features + [target]]
print(reduced_df.shape)


TypeError: DataFrame.sort_values() missing 1 required positional argument: 'by'

In [ ]:
# Convert target to numeric if it's not
accepted_checked['target_default'] = accepted_checked['target_default'].astype(int)


In [ ]:
# Numeric columns
numeric_cols = accepted_checked.select_dtypes(include=['float64', 'int']).columns.tolist()
numeric_cols = [col for col in numeric_cols if col != 'target_default']  # exclude target

# Compute correlation with target
numeric_corr = accepted_checked[numeric_cols + ['target_default']].corr()['target_default'].sort_values(ascending=False)

print("Top numeric features correlated with target:\n", numeric_corr.head(20))


Top numeric features correlated with target:
 target_default          1.000000
int_rate                0.309656
acc_open_past_24mths    0.134946
dti                     0.115809
num_tl_op_past_12m      0.113838
inq_last_6mths          0.090889
num_rev_tl_bal_gt_0     0.077819
num_actv_rev_tl         0.076995
loan_amnt               0.071889
funded_amnt             0.071889
funded_amnt_inv         0.071691
percent_bc_gt_75        0.069596
bc_util                 0.065129
revol_util              0.054838
num_op_rev_tl           0.047112
num_sats                0.043987
open_acc                0.043922
num_actv_bc_tl          0.043615
installment             0.036419
pub_rec_bankruptcies    0.032589
Name: target_default, dtype: float64


In [ ]:
# Set correlation threshold
corr_threshold = 0.05

# Keep numeric features with correlation above threshold
numeric_corr = numeric_corr.drop(labels=['target_default'])  # remove target itself
selected_numeric = numeric_corr[abs(numeric_corr) > corr_threshold].index.tolist()

# Drop ID-like or NaN correlated features
selected_numeric = [col for col in selected_numeric if col not in ['id', 'member_id', 'policy_code']]

# Reduce boolean columns by variance
bool_cols = accepted_checked.select_dtypes(include=['bool']).columns.tolist()
selected_bool = [col for col in bool_cols if accepted_checked[col].mean() > 0.01]  # keep only if >1% True

# Combine
selected_features = selected_numeric + selected_bool + ['target_default']

# Create reduced dataframe
reduced_df = accepted_checked[selected_features]
print("Original shape:", accepted_checked.shape)
print("Reduced shape:", reduced_df.shape)


Original shape: (229565, 2008)
Reduced shape: (229565, 118)


In [ ]:
reduced_df.head()
reduced_df.describe()


,int_rate,acc_open_past_24mths,dti,num_tl_op_past_12m,inq_last_6mths,num_rev_tl_bal_gt_0,num_actv_rev_tl,loan_amnt,funded_amnt,funded_amnt_inv,...,tot_cur_bal,mo_sin_rcnt_rev_tl_op,mo_sin_rcnt_tl,tot_hi_cred_lim,avg_cur_bal,total_bc_limit,bc_open_to_buy,fico_range_high,fico_range_low,target_default
count,229565.000000,229565.000000,229565.000000,229565.000000,229565.000000,229565.000000,229565.000000,229565.000000,229565.000000,229565.00000,...,2.295650e+05,229565.000000,229565.000000,2.295650e+05,229565.000000,229565.000000,229565.000000,229565.000000,229565.000000,229565.000000
mean,12.228146,4.754453,18.970341,2.217546,0.581826,5.721094,5.780476,14599.084682,14599.084682,14591.46775,...,1.387909e+05,13.277381,7.879407,1.724822e+05,13011.294967,21901.212175,9872.878427,698.191301,694.191166,0.204169
std,4.275980,3.256490,9.060414,1.896280,0.876856,3.318321,3.432095,8584.898738,8584.898738,8580.48830,...,1.564170e+05,16.899160,9.054215,1.768672e+05,15764.102448,21936.390844,14953.134903,30.841101,30.840439,0.403094
min,5.320000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1000.000000,1000.000000,900.00000,...,0.000000e+00,0.000000,0.000000,2.500000e+03,0.000000,0.000000,0.000000,664.000000,660.000000,0.000000
25%,9.170000,2.000000,12.390000,1.000000,0.000000,3.000000,3.000000,8000.000000,8000.000000,8000.00000,...,3.014700e+04,4.000000,3.000000,5.018300e+04,3128.000000,7800.000000,1469.000000,674.000000,670.000000,0.000000
50%,12.050000,4.000000,18.410000,2.000000,0.000000,5.000000,5.000000,12350.000000,12350.000000,12325.00000,...,7.709100e+04,8.000000,5.000000,1.089000e+05,7102.000000,15000.000000,4669.000000,689.000000,685.000000,0.000000
75%,14.650000,6.000000,25.150000,3.000000,1.000000,7.000000,7.000000,20000.000000,20000.000000,20000.00000,...,2.051980e+05,16.000000,10.000000,2.480640e+05,17836.000000,28400.000000,11819.000000,714.000000,710.000000,0.000000
max,28.990000,64.000000,999.000000,30.000000,5.000000,45.000000,52.000000,35000.000000,35000.000000,35000.00000,...,3.726495e+06,324.000000,263.000000,9.999999e+06,478909.000000,344200.000000,263953.000000,850.000000,845.000000,1.000000


In [ ]:
# Set output path
output_path = "/content/drive/MyDrive/LOAN_DEFAULT_RISK/data/processed/reduced_accepted.csv"

# Save to CSV
reduced_df.to_csv(output_path, index=False)

print(f"Reduced dataset saved successfully at: {output_path}")


Reduced dataset saved successfully at: /content/drive/MyDrive/LOAN_DEFAULT_RISK/data/processed/reduced_accepted.csv


In [ ]:
# Save as Parquet
reduced_df.to_parquet("/content/drive/MyDrive/LOAN_DEFAULT_RISK/data/processed/reduced_accepted.parquet", index=False)


In [ ]:
import pandas as pd

# Paths
PROCESSED_PATH = "/content/drive/MyDrive/LOAN_DEFAULT_RISK/data/processed/"

# Load main reduced dataset
accepted_df = pd.read_csv(PROCESSED_PATH + "reduced_accepted.csv")

# Load aggregated rejected dataset
rejected_df = pd.read_csv(PROCESSED_PATH + "rejected_monthly_clean.csv")

# Load unemployment dataset
unemp_df = pd.read_csv(PROCESSED_PATH + "ca_unemployment_clean.csv")

# Ensure year_month columns are period type
accepted_df['year_month'] = pd.to_datetime(accepted_df['year_month']).dt.to_period('M')
rejected_df['year_month'] = pd.to_datetime(rejected_df['year_month']).dt.to_period('M')
unemp_df['year_month'] = pd.to_datetime(unemp_df['year_month']).dt.to_period('M')

print("Accepted shape:", accepted_df.shape)
print("Rejected shape:", rejected_df.shape)
print("Unemployment shape:", unemp_df.shape)


KeyError: 'year_month'

In [ ]:
print("Accepted columns:", accepted_df.columns.tolist()[:20])
print("Rejected columns:", rejected_df.columns.tolist())
print("Unemployment columns:", unemp_df.columns.tolist())


Accepted columns: ['int_rate', 'acc_open_past_24mths', 'dti', 'num_tl_op_past_12m', 'inq_last_6mths', 'num_rev_tl_bal_gt_0', 'num_actv_rev_tl', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv', 'percent_bc_gt_75', 'bc_util', 'revol_util', 'mo_sin_old_rev_tl_op', 'mths_since_recent_inq', 'total_rev_hi_lim', 'mths_since_recent_bc', 'mort_acc', 'tot_cur_bal', 'mo_sin_rcnt_rev_tl_op']
Rejected columns: ['year_month', 'rejected_applications', 'rejected_avg_amount', 'rejected_avg_dti', 'rejected_avg_risk_score']
Unemployment columns: ['year_month', 'unemployment_rate']


In [ ]:
import pandas as pd

# Load reduced accepted dataset
accepted_df = pd.read_csv(PROCESSED_PATH + "reduced_accepted.csv")

# Check if 'year_month' exists
if 'year_month' not in accepted_df.columns:
    # Create 'year_month' from the original 'issue_d' column if available
    if 'issue_d' in accepted_df.columns:
        accepted_df['year_month'] = pd.to_datetime(accepted_df['issue_d'], errors='coerce').dt.to_period('M')
    else:
        # If 'issue_d' not available, you must have some date column to derive it
        raise ValueError("No date column available to create 'year_month' in accepted_df")

# Ensure 'year_month' is period type for merging
accepted_df['year_month'] = accepted_df['year_month'].astype('period[M]')

# Convert rejected and unemployment columns to period[M] as well
rejected_df['year_month'] = rejected_df['year_month'].astype('period[M]')
unemp_df['year_month'] = unemp_df['year_month'].astype('period[M]')

# Merge accepted with rejected
merged_df = pd.merge(
    accepted_df,
    rejected_df,
    how='left',
    on='year_month'
)

# Merge with unemployment
merged_df = pd.merge(
    merged_df,
    unemp_df,
    how='left',
    on='year_month'
)

print("Merged dataset shape:", merged_df.shape)
print(merged_df.head())


ValueError: No date column available to create 'year_month' in accepted_df

# only colfornia data from all three datasets

### STEP 1: Load the accepted dataset

In [4]:
import pandas as pd

accepted_path = PROCESSED_PATH + "accepted_clean_model.csv"

accepted_df = pd.read_csv(accepted_path, low_memory=False)

print("Shape:", accepted_df.shape)
accepted_df.head()


Shape: (229565, 2008)


,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,annual_inc,dti,delinq_2yrs,fico_range_low,fico_range_high,...,earliest_cr_line_Sep-2008,earliest_cr_line_Sep-2009,earliest_cr_line_Sep-2010,earliest_cr_line_Sep-2011,earliest_cr_line_Sep-2012,earliest_cr_line_Sep-2013,earliest_cr_line_Sep-2014,earliest_cr_line_Sep-2015,initial_list_status_w,application_type_Joint App
0,3600.0,3600.0,3600.0,13.99,123.03,55000.0,5.91,0.0,675.0,679.0,...,False,False,False,False,False,False,False,False,True,False
1,24700.0,24700.0,24700.0,11.99,820.28,65000.0,16.06,1.0,715.0,719.0,...,False,False,False,False,False,False,False,False,True,False
2,20000.0,20000.0,20000.0,10.78,432.66,63000.0,10.78,0.0,695.0,699.0,...,False,False,False,False,False,False,False,False,True,True
3,10400.0,10400.0,10400.0,22.45,289.91,104433.0,25.37,1.0,695.0,699.0,...,False,False,False,False,False,False,False,False,True,False
4,11950.0,11950.0,11950.0,13.44,405.18,34000.0,10.20,0.0,690.0,694.0,...,False,False,False,False,False,False,False,False,True,False


### ✅ STEP 2: Identify the state column

In [5]:
[col for col in accepted_df.columns if "state" in col.lower() or "addr" in col.lower()]


['addr_state_AL',
 'addr_state_AR',
 'addr_state_AZ',
 'addr_state_CA',
 'addr_state_CO',
 'addr_state_CT',
 'addr_state_DC',
 'addr_state_DE',
 'addr_state_FL',
 'addr_state_GA',
 'addr_state_HI',
 'addr_state_IA',
 'addr_state_ID',
 'addr_state_IL',
 'addr_state_IN',
 'addr_state_KS',
 'addr_state_KY',
 'addr_state_LA',
 'addr_state_MA',
 'addr_state_MD',
 'addr_state_ME',
 'addr_state_MI',
 'addr_state_MN',
 'addr_state_MO',
 'addr_state_MS',
 'addr_state_MT',
 'addr_state_NC',
 'addr_state_ND',
 'addr_state_NE',
 'addr_state_NH',
 'addr_state_NJ',
 'addr_state_NM',
 'addr_state_NV',
 'addr_state_NY',
 'addr_state_OH',
 'addr_state_OK',
 'addr_state_OR',
 'addr_state_PA',
 'addr_state_RI',
 'addr_state_SC',
 'addr_state_SD',
 'addr_state_TN',
 'addr_state_TX',
 'addr_state_UT',
 'addr_state_VA',
 'addr_state_VT',
 'addr_state_WA',
 'addr_state_WI',
 'addr_state_WV',
 'addr_state_WY']

In [6]:
accepted_df['addr_state'].value_counts().head(10)


KeyError: 'addr_state'

### ✅ STEP 3: Filter only California (CA)

In [7]:
accepted_ca_df = accepted_df[accepted_df['addr_state_CA'] == 1].copy()

print("California-only shape:", accepted_ca_df.shape)
accepted_ca_df.head()


California-only shape: (32316, 2008)


,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,annual_inc,dti,delinq_2yrs,fico_range_low,fico_range_high,...,earliest_cr_line_Sep-2008,earliest_cr_line_Sep-2009,earliest_cr_line_Sep-2010,earliest_cr_line_Sep-2011,earliest_cr_line_Sep-2012,earliest_cr_line_Sep-2013,earliest_cr_line_Sep-2014,earliest_cr_line_Sep-2015,initial_list_status_w,application_type_Joint App
10,18000.0,18000.0,18000.0,19.48,471.70,150000.0,9.39,0.0,665.0,669.0,...,False,False,False,False,False,False,False,False,True,False
62,20200.0,20200.0,20200.0,18.49,518.35,60000.0,34.84,0.0,720.0,724.0,...,False,False,False,False,False,False,False,False,True,False
96,12000.0,12000.0,12000.0,9.17,382.55,39400.0,26.32,0.0,705.0,709.0,...,False,False,False,False,False,False,False,False,True,False
105,15975.0,15975.0,15975.0,16.59,566.30,58800.0,20.45,0.0,665.0,669.0,...,False,False,False,False,False,False,False,False,True,False
114,22875.0,22875.0,22875.0,15.77,553.49,137000.0,32.30,0.0,700.0,704.0,...,False,False,False,False,False,False,False,False,True,False


### ✅ STEP 3: Filter only California (CA)

Once confirmed, extract California:

In [8]:
accepted_ca_df = accepted_df[accepted_df['addr_state_CA'] == 1].copy()

print("California-only shape:", accepted_ca_df.shape)
accepted_ca_df.head()


California-only shape: (32316, 2008)


,loan_amnt,funded_amnt,funded_amnt_inv,int_rate,installment,annual_inc,dti,delinq_2yrs,fico_range_low,fico_range_high,...,earliest_cr_line_Sep-2008,earliest_cr_line_Sep-2009,earliest_cr_line_Sep-2010,earliest_cr_line_Sep-2011,earliest_cr_line_Sep-2012,earliest_cr_line_Sep-2013,earliest_cr_line_Sep-2014,earliest_cr_line_Sep-2015,initial_list_status_w,application_type_Joint App
10,18000.0,18000.0,18000.0,19.48,471.70,150000.0,9.39,0.0,665.0,669.0,...,False,False,False,False,False,False,False,False,True,False
62,20200.0,20200.0,20200.0,18.49,518.35,60000.0,34.84,0.0,720.0,724.0,...,False,False,False,False,False,False,False,False,True,False
96,12000.0,12000.0,12000.0,9.17,382.55,39400.0,26.32,0.0,705.0,709.0,...,False,False,False,False,False,False,False,False,True,False
105,15975.0,15975.0,15975.0,16.59,566.30,58800.0,20.45,0.0,665.0,669.0,...,False,False,False,False,False,False,False,False,True,False
114,22875.0,22875.0,22875.0,15.77,553.49,137000.0,32.30,0.0,700.0,704.0,...,False,False,False,False,False,False,False,False,True,False


In [9]:
state_cols = [c for c in accepted_ca_df.columns if c.startswith("addr_state_")]

accepted_ca_df.drop(columns=state_cols, inplace=True)

print("Dropped state dummy columns.")
print("New shape:", accepted_ca_df.shape)


Dropped state dummy columns.
New shape: (32316, 1958)


### ✅ STEP 4: Save California-only accepted dataset

Because your file is large, I strongly recommend Parquet (much faster than CSV):

In [10]:
accepted_ca_df.to_parquet(PROCESSED_PATH + "accepted_CA_clean_model.parquet", index=False)


In [11]:
# should return only 1s
accepted_ca_df['addr_state_CA'].value_counts()


KeyError: 'addr_state_CA'

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
DATA_PATH = "/content/drive/MyDrive/LOAN_DEFAULT_RISK/data/filtered/"

df = pd.read_csv(DATA_PATH + "full_CA_merged.csv", low_memory=False)

print(df.shape)
df.head()


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/LOAN_DEFAULT_RISK/data/filtered/full_CA_merged.csv'

In [4]:
import os

FILTERED_PATH = "/content/drive/MyDrive/LOAN_DEFAULT_RISK/data/filtered"

files = os.listdir(FILTERED_PATH)

print("Files in filtered folder:\n")
for f in files:
    print(f)


Files in filtered folder:

ca_unemployment_clean.csv
accepted_CA.csv
rejected_CA.csv
full_CA_dataset.csv
full_CA_merged_sample.csv


In [5]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
DATA_PATH = "/content/drive/MyDrive/LOAN_DEFAULT_RISK/data/filtered/full_CA_dataset.csv"
df = pd.read_csv(DATA_PATH, low_memory=False)

print("Shape:", df.shape)
df.head()


In [7]:
df_model = df[df['loan_status'] == 'Accepted'].copy()

print(df_model['target_default'].value_counts(normalize=True))


Series([], Name: proportion, dtype: float64)


In [8]:
y = df_model['target_default']


In [9]:
leakage_cols = [
    'loan_status','target_default','recoveries','collection_recovery_fee',
    'last_pymnt_d','last_pymnt_amnt','next_pymnt_d','out_prncp','out_prncp_inv'
]

date_cols = ['Application Date','issue_d','earliest_cr_line','last_credit_pull_d']

df_model = df_model.drop(columns=[c for c in leakage_cols+date_cols if c in df_model.columns], errors='ignore')


In [10]:
X = df_model


In [11]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer


In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)


ValueError: With n_samples=0, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

In [13]:
df.columns


Index(['id', 'member_id', 'loan_amnt', 'funded_amnt', 'funded_amnt_inv',
       'term', 'int_rate', 'installment', 'grade', 'sub_grade',
       ...
       'Loan Title', 'Risk_Score', 'Debt-To-Income Ratio', 'Zip Code', 'State',
       'Employment Length', 'Policy Code', 'year', 'month',
       'unemployment_rate'],
      dtype='object', length=164)